# Supervised fine-tuning

`answer_only` accepts one or more prepared training files and a separate real dev set. This notebook can now train either the original **gold-only** data or **gold + verified synthetic** data for Qwen2.5-0.5B and Qwen2.5-3B.

The split-first gold data contains 1,137 supported training questions expanded to 1,363 answer-alias rows. The verified synthetic addition contributes 652 training questions/rows. The combined variant therefore contains **1,789 unique questions and 2,015 SFT rows**. The 155-question synthetic validation partition is excluded from optimization, and the same 160 unfiltered real BioASQ dev questions remain the checkpoint-selection set for a controlled comparison.

Set `TRAINING_DATA_VARIANT` to `"gold_plus_synthetic"` or `"gold_only"`. The current full combined-data recipe keeps all 2,015 rows and uses 4-bit LoRA with rank/alpha 32, dropout 0.05, batch size 1, accumulation 32, learning rate 3e-4, 20 warmup steps, evaluation every 40 optimizer steps, up to 6 epochs, and generated-dev checkpoint selection with patience 3. The wider evaluation interval preserves approximately the gold-only epoch cadence after the dataset increase.

Model loading may download weights unless `local_files_only=True`. Long contexts can be truncated, so source-level evidence eligibility does not guarantee that every matching span survives tokenization.

The evidence and rationale presets remain available for their specialized workflows.

Execution is enabled for the two configured runs. Select the project's `.venv` kernel and inspect both plans before running the execution cell; the models train sequentially.


In [1]:
from pathlib import Path
import json
import re
import sys

PROJECT_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "src/notebook_workflows").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook_workflows import describe, plan, execute
from src.notebook_workflows.presets import PRESETS, configuration
print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)


Project: C:\Users\Dustin\Projects\BioASQ-MRES
Python: c:\Users\Dustin\Projects\BioASQ-MRES\.venv\Scripts\python.exe


## Choose a preset and training-data variant

- **answer_only**: trains on `gold_only` or `gold_plus_synthetic`
- **evidence_plus_answer** / **evidence_answer_control**: matched evidence experiments
- **rationale_from_base** / **rationale_continued**: rationale experiments

The current configuration prepares combined-data runs for both Qwen sizes. Input counts and train/dev disjointness are checked before plans are created.


In [2]:
CATEGORY = "sft"
PRESET = "answer_only"
TRAINING_DATA_VARIANT = "gold_plus_synthetic"  # or "gold_only"

DATA = (
    PROJECT_ROOT
    / "data/BioASQ_factoid_sft_prepared"
    / "split_first_train90_dev10_supported_train_seed3407"
)
SYNTHETIC = (
    PROJECT_ROOT
    / "Artifacts/notebook_runs/synthetic/synthetic_qa"
    / "20260928-031536-dcd26860/synthetic/training_ready"
)

TRAIN_INPUTS = {
    "gold_only": [str(DATA / "train_prepared.json")],
    "gold_plus_synthetic": [
        str(DATA / "train_prepared.json"),
        str(SYNTHETIC / "sft_train.json"),
    ],
}
if TRAINING_DATA_VARIANT not in TRAIN_INPUTS:
    raise ValueError(f"Unknown TRAINING_DATA_VARIANT: {TRAINING_DATA_VARIANT}")

OVERRIDES = {
    "train_input": TRAIN_INPUTS[TRAINING_DATA_VARIANT],
    "eval_input": [str(DATA / "dev_prepared.json")],

    # Retuned schedule for the 2,015-row combined dataset.
    "learning_rate": 3e-4,
    "warmup_steps": 20,
    "save_steps": 40,
    "num_train_epochs": 6.0,
    "early_stopping_patience": 3,
    "early_stopping_threshold": 0.001,
}

def original_question_id(row):
    value = row.get("source_question_id") or row.get("original_id") or row.get("id")
    return re.sub(r"__(?:supported_alias|alias)_.*$", "", str(value))

def prepared_input_summary(paths):
    rows = []
    for path in paths:
        rows.extend(json.loads(Path(path).read_text(encoding="utf-8")))
    return {
        "rows": len(rows),
        "unique_questions": len({original_question_id(row) for row in rows}),
        "ids": {original_question_id(row) for row in rows},
    }

train_summary = prepared_input_summary(OVERRIDES["train_input"])
dev_summary = prepared_input_summary(OVERRIDES["eval_input"])
overlap = train_summary["ids"] & dev_summary["ids"]
if overlap:
    raise ValueError(f"Training/dev overlap: {len(overlap)} question IDs")
print(json.dumps({
    "training_data_variant": TRAINING_DATA_VARIANT,
    "training_rows": train_summary["rows"],
    "unique_training_questions": train_summary["unique_questions"],
    "real_dev_rows": dev_summary["rows"],
    "train_dev_overlap": len(overlap),
}, indent=2))

VARIANTS = [
    {
        "label": f"qwen05b_{TRAINING_DATA_VARIANT}",
        "overrides": {"model_name": "Qwen/Qwen2.5-0.5B-Instruct"},
    },
    {
        "label": f"qwen3b_{TRAINING_DATA_VARIANT}",
        "overrides": {"model_name": "Qwen/Qwen2.5-3B-Instruct"},
    },
]

RUN = True
ALLOW_API = False

METHOD, PARAMETERS = configuration(CATEGORY, PRESET, OVERRIDES)
print(json.dumps({"method": METHOD, "parameters": PARAMETERS}, indent=2))


{
  "training_data_variant": "gold_plus_synthetic",
  "training_rows": 2015,
  "unique_training_questions": 1789,
  "real_dev_rows": 160,
  "train_dev_overlap": 0
}
{
  "method": "answer_sft",
  "parameters": {
    "chat_template": "qwen-2.5",
    "prompt_format": "chat",
    "prompt_file": "prompts/factoid_single_answer_aligned.json",
    "prompt": "factoid-single-answer-extractive-v1",
    "max_resources": 0,
    "max_resource_chars": 0,
    "max_seq_length": 4096,
    "train_input": [
      "C:\\Users\\Dustin\\Projects\\BioASQ-MRES\\data\\BioASQ_factoid_sft_prepared\\split_first_train90_dev10_supported_train_seed3407\\train_prepared.json",
      "C:\\Users\\Dustin\\Projects\\BioASQ-MRES\\Artifacts\\notebook_runs\\synthetic\\synthetic_qa\\20260928-031536-dcd26860\\synthetic\\training_ready\\sft_train.json"
    ],
    "eval_input": [
      "C:\\Users\\Dustin\\Projects\\BioASQ-MRES\\data\\BioASQ_factoid_sft_prepared\\split_first_train90_dev10_supported_train_seed3407\\dev_prepared.json

## Inspect and preview

Options are read from source without importing the training stack. Unknown options and invalid choices are rejected. Outputs go under Artifacts/notebook_runs and cannot overwrite earlier runs. Rerun this cell to allocate fresh plans.


In [3]:
print(json.dumps(describe(METHOD), indent=2, default=str))
PLANS = []
for variant in VARIANTS:
    method, parameters = configuration(CATEGORY, PRESET, {**OVERRIDES, **variant["overrides"]})
    preview = plan(method, parameters, project_root=PROJECT_ROOT)
    PLANS.append(preview)
    print("\nVariant:", variant["label"])
    print("Method:", preview["method"])
    print("Output:", preview["run_dir"])
    print("Requires GPU:", preview["gpu"], "| Calls API:", preview["api"])
    print("Unresolved inputs:", preview["missing"] or "none")
    print(json.dumps(preview["parameters"], indent=2))


{
  "method": "answer_sft",
  "description": "Answer-only LoRA SFT for any supported model, using explicit train/dev files.",
  "defaults": {
    "chat_template": "qwen-2.5",
    "prompt_format": "chat",
    "prompt_file": "prompts/factoid_single_answer_aligned.json",
    "prompt": "factoid-single-answer-extractive-v1",
    "max_resources": 0,
    "max_resource_chars": 0,
    "max_seq_length": 4096,
    "train_input": null,
    "eval_input": null,
    "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
    "question_types": [
      "factoid"
    ],
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 32,
    "warmup_steps": 5,
    "num_train_epochs": 6.0,
    "learning_rate": 0.0006,
    "weight_decay": 0.01,
    "lora_r": 32,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "dataset_num_proc": 1,
    "selection_metric": "generated_primary_score",
    "selection_max_new_tokens": 64,
    "selection_max_seq_length": 4096,
    "save_stra

## Execute reviewed plans

Set RUN=True, then rerun configuration and preview. API methods also need ALLOW_API=True. Plans cannot be changed after preview. Each run retains its configuration, input hashes, status and log. A repeated execution of the same plan refuses to overwrite its directory; create a fresh plan instead.


In [4]:
RESULTS = []
for preview in PLANS:
    if RUN:
        result = execute(preview, run=True, allow_api=ALLOW_API)
        RESULTS.append(str(result))
    else:
        print("Preview only:", preview["method"], "- set RUN=True to execute.")


W0929 10:01:45.643145 11008 .venv\Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0929 10:01:45.739663 11008 .venv\Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
c:\Users\Dustin\Projects\BioASQ-MRES\.venv\lib\site-packages\unsloth\__init__.py:1589: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *
🦥

KeyboardInterrupt: 

## Inspect outputs

Use these explicit paths in the next notebook. Historical commands, plots and outputs are preserved in reproducibility/notebook_archive.zip; notebooks/MIGRATION.md maps old notebooks to these workflows.


In [ ]:
for run in RESULTS:
    path = Path(run)
    print("\nRun:", path)
    print((path / "status.json").read_text(encoding="utf-8"))
    print("Log:", path / "run.log")
    print("Outputs:", [p.name for p in path.iterdir()
                      if p.name not in {"run.log", "plan.json", "input_hashes.json"}])



Run: C:\Users\Dustin\Projects\BioASQ-MRES\Artifacts\notebook_runs\sft\answer_sft\20260928-135402-3d75d795
{
  "status": "complete",
  "method": "answer_sft",
  "fingerprint": "a939650b4801675bd171089897d98c802a3bd5d4b67e1e75d28e4ec4694aa0c4"
}

Log: C:\Users\Dustin\Projects\BioASQ-MRES\Artifacts\notebook_runs\sft\answer_sft\20260928-135402-3d75d795\run.log
Outputs: ['adapter', 'adapter_best_eval_loss', 'eval_loss_selection', 'evidence_mrr_selection', 'generated_dev_selection', 'managed', 'model_registry.json', 'status.json', 'trainer_output', 'trainer_prepared']

Run: C:\Users\Dustin\Projects\BioASQ-MRES\Artifacts\notebook_runs\sft\answer_sft\20260928-135402-88eb27d2
{
  "status": "complete",
  "method": "answer_sft",
  "fingerprint": "ba10cea3112333b9a0f228273eb8ede26df94d89c18371b3607c07f8fe2b91a2"
}

Log: C:\Users\Dustin\Projects\BioASQ-MRES\Artifacts\notebook_runs\sft\answer_sft\20260928-135402-88eb27d2\run.log
Outputs: ['adapter', 'adapter_best_eval_loss', 'eval_loss_selection', 

In [ ]:
from IPython.display import Image, display

for run in RESULTS:
    plot = Path(run) / "adapter" / "training_curves.png"
    print(run)
    if plot.exists():
        display(Image(filename=str(plot)))

C:\Users\Dustin\Projects\BioASQ-MRES\Artifacts\notebook_runs\sft\answer_sft\20260928-135402-3d75d795
C:\Users\Dustin\Projects\BioASQ-MRES\Artifacts\notebook_runs\sft\answer_sft\20260928-135402-88eb27d2
